[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/225/blob/main/06_pandas_intro/06_5_GroupBy.ipynb)

# 06.5: GroupBy and Aggregation

The most revealing questions in the Titanic data are comparisons between groups: did women survive at a higher rate than men? Were first-class passengers more likely to survive than third-class? To answer these, you need to compute a summary separately for each group and then compare the results. That is what **GroupBy** does. Let's load the data.

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_titanic.csv"
df = pd.read_csv(url)
df.columns = df.columns.str.lower()
df = df.rename(columns={
    'siblings/spouses aboard': 'sibsp',
    'parents/children aboard': 'parch'
})
df.head()

,survived,pclass,name,sex,age,sibsp,parch,fare
0,0,3,Mr. Owen Harris Braund,male,22.0,1,0,7.2500
1,1,1,Mrs. John Bradley (Florence Briggs Thayer) Cum...,female,38.0,1,0,71.2833
2,1,3,Miss. Laina Heikkinen,female,26.0,0,0,7.9250
3,1,1,Mrs. Jacques Heath (Lily May Peel) Futrelle,female,35.0,1,0,53.1000
4,0,3,Mr. William Henry Allen,male,35.0,0,0,8.0500


Eight columns, 887 rows. Every analysis so far has worked on the full dataset or on a filtered subset. GroupBy lets us split the dataset into groups, compute a summary within each group, and combine the results -- answering comparison questions in a single call instead of multiple separate filters.

## 1. The Split–Apply–Combine Pattern

GroupBy works in three steps that always happen in the same order:

**Split** — divide the DataFrame into groups based on the values in one column. If you group by `sex`, you get two groups: all male rows and all female rows.

**Apply** — compute something within each group. For example, compute the mean of the `survived` column within each group.

**Combine** — assemble the per-group results into a single output Series or DataFrame.

```
Full dataset (887 rows)
        sex   survived   fare
        male       0     7.25
        female     1    71.28
        female     1     7.92
        male       0    53.10
        ...

      ▼  SPLIT by "sex"

  GROUP: male         GROUP: female
  survived  fare      survived  fare
      0     7.25          1    71.28
      0    53.10          1     7.92
      ...                ...

      ▼  APPLY: mean("survived")

  male → 0.19          female → 0.74

      ▼  COMBINE

  sex       survived
  female    0.74
  male      0.19
```

## 2. Your First GroupBy

The syntax is: `df.groupby("column")["column_to_aggregate"].aggregation_function()`

Let's answer the first question: did women survive at a higher rate than men?

In [2]:
# .mean() on a 0/1 column gives the proportion that equals 1 — the survival rate
df.groupby("sex")["survived"].mean()

sex
female    0.742038
male      0.190227
Name: survived, dtype: float64

Women survived at 74%, men at 19%. That's a dramatic difference, and it's not a coincidence — "women and children first" was the evacuation protocol.

Now the second question: did survival differ by class?

In [3]:
df.groupby("pclass")["survived"].mean().round(2)

pclass
1    0.63
2    0.47
3    0.24
Name: survived, dtype: float64

First-class passengers survived at 63%; third-class at only 24%. Part of this is proximity — third-class cabins were below decks, and by the time passengers made it to the top the lifeboats were gone. Part of it may also reflect who was directed toward the lifeboats first.

Let's also check average fare by class — does the fare data corroborate the class labels?

In [4]:
df.groupby("pclass")["fare"].mean().round(2)

pclass
1    84.15
2    20.66
3    13.71
Name: fare, dtype: float64

First-class passengers paid an average of £84; third-class paid £14. The fare values corroborate the class labels and suggest that class is a genuine socioeconomic signal, not just a ticket category.

## 3. When You Need More Than One Number Per Group

A survival rate is useful, but it's just one number. To understand a group properly you often want several summaries at once — how many passengers were in each group? What was the youngest? The oldest? The average fare?

`.agg()` computes multiple aggregations in a single call.

In [5]:
# How many passengers per class, and what was their average fare?
df.groupby("pclass")["fare"].agg(["count", "mean", "median", "min", "max"]).round(2)

,count,mean,median,min,max
pclass,,,,,
1,216,84.15,60.29,0.0,512.33
2,184,20.66,14.25,0.0,73.50
3,487,13.71,8.05,0.0,69.55


That table shows 216 first-class passengers with a mean fare of £84 and a median of £60; 487 third-class passengers with a mean of £14 and a median of £8. The gap between mean and median in first class hints at a few very expensive tickets pulling the average up. Named aggregations let you give each output column a clear label.

In [6]:
# Named aggregations — give each output column a meaningful name
df.groupby("pclass").agg(
    passengers    = ("survived", "count"),
    survival_rate = ("survived", "mean"),
    avg_fare      = ("fare",     "mean"),
    avg_age       = ("age",      "mean"),
).round(2)

,passengers,survival_rate,avg_fare,avg_age
pclass,,,,
1,216,0.63,84.15,38.79
2,184,0.47,20.66,29.87
3,487,0.24,13.71,25.19


Reading this table: first-class passengers were older on average (39 years vs 25 for third class), paid more, and survived at a much higher rate. These three facts together suggest that class, age, and survival are all correlated — something we'll explore further in the next notebook.

## 4. Grouping by Two Keys

The next natural question: is the gender survival gap consistent across all three classes, or is it stronger in some classes than others?

To answer that, you need to group by *both* class and sex simultaneously.

In [7]:
df.groupby(["pclass", "sex"])["survived"].mean().round(2)

pclass  sex   
1       female    0.97
        male      0.37
2       female    0.92
        male      0.16
3       female    0.50
        male      0.14
Name: survived, dtype: float64

Every sex/class combination shows the same pattern: women survived at much higher rates than men. Third-class women (50%) survived at more than double the rate of first-class men (37%). Being female was the dominant factor, more important than class.

In [8]:
# Full picture: count and survival rate by class and sex
df.groupby(["pclass", "sex"]).agg(
    count         = ("survived", "count"),
    survival_rate = ("survived", "mean"),
).round(2)

count  survival_rate
pclass sex                         
1      female     94           0.97
       male      122           0.37
2      female     76           0.92
       male      108           0.16
3      female    144           0.50
       male      343           0.14

Reading the counts: 94 first-class women, 122 first-class men, 343 third-class men -- and those 343 men had only a 14% survival rate. The raw numbers help you see whether a percentage is based on a large sample or a small one.

## 5. Cleaning Up the Result: `.reset_index()`

After a groupby, the grouping column (`pclass`, `sex`) becomes the **index** of the result rather than a regular column. This is fine for quick inspection but awkward when you want to sort, rename, or filter the result further. `.reset_index()` promotes the index back to a regular column.

In [9]:
# Without reset_index: pclass is the index
result = df.groupby("pclass")["survived"].mean()
print("Type:", type(result))
print("Index:", result.index.tolist())
print(result)

Type: <class 'pandas.Series'>
Index: [1, 2, 3]
pclass
1    0.629630
2    0.472826
3    0.244353
Name: survived, dtype: float64


With `pclass` as a regular column you can chain further operations. Here we rename the `survived` column to `survival_rate` and sort descending so the highest-survival class appears first.

In [10]:
# With reset_index: pclass is a regular column you can sort, rename, filter
result_df = (
    df.groupby("pclass")["survived"].mean()
    .reset_index()
    .rename(columns={"survived": "survival_rate"})
    .sort_values("survival_rate", ascending=False)
)
result_df

,pclass,survival_rate
0,1,0.629630
1,2,0.472826
2,3,0.244353


## 6. Putting It Together

Let's answer the compound question: *For each combination of class and sex, how many passengers were there, what fraction survived, how old were they on average, and what did they pay?*

In [11]:
summary = (
    df.groupby(["pclass", "sex"])
    .agg(
        passengers    = ("survived", "count"),
        survival_rate = ("survived", "mean"),
        avg_age       = ("age",      "mean"),
        avg_fare      = ("fare",     "mean"),
    )
    .round(2)
    .reset_index()
    .sort_values(["pclass", "sex"])
)
summary

,pclass,sex,passengers,survival_rate,avg_age,avg_fare
0,1,female,94,0.97,35.26,106.13
1,1,male,122,0.37,41.51,67.23
2,2,female,76,0.92,28.98,21.97
3,2,male,108,0.16,30.49,19.74
4,3,female,144,0.50,22.14,16.12
5,3,male,343,0.14,26.47,12.70


The summary table has one row per class/sex combination. First-class women paid over £106 on average and survived at 97%. Third-class men paid £13 and survived at 14%. Every column tells part of the same story: class and sex were the two strongest predictors of survival. Here is the gender gap across all three classes in a compact format.

In [12]:
# Highlight the gender gap in a readable format
print("Survival rates by class and sex\n")
for cls in sorted(df["pclass"].unique()):
    male   = df.loc[(df["pclass"]==cls) & (df["sex"]=="male"),   "survived"].mean()
    female = df.loc[(df["pclass"]==cls) & (df["sex"]=="female"), "survived"].mean()
    print(f"Class {cls}:  men {male:.0%}  |  women {female:.0%}")

Survival rates by class and sex

Class 1:  men 37%  |  women 97%
Class 2:  men 16%  |  women 92%
Class 3:  men 14%  |  women 50%


The gender gap narrows from 60 percentage points in first class (97% vs 37%) to 36 points in third class (50% vs 14%), but it never reverses. Even third-class women outsurvived first-class men.

## What's next

GroupBy gives you one row of summary statistics per group. But many questions are inherently two-dimensional: how does survival vary by both class and sex at the same time? Answering that question with multiple separate GroupBy calls produces separate tables that are hard to compare. Notebook 06.6 introduces crosstabs and pivot tables, which arrange the same information into a single grid and make cross-group comparisons immediate.